# 0. Problem
## 1527. Patients With a Condition — Easy
Return patients whose space-separated conditions contain a condition code that starts with `DIAB1`.

Official: https://leetcode.com/problems/patients-with-a-condition/

# 1. Setup

In [ ]:
import pandas as pd
patients_rows=[(1,"Daniel","YFEV COUGH"),(2,"Alice",None),(3,"Bob","DIAB100 MYOP"),(4,"George","ACNE DIAB100"),(5,"Alain","DIAB201")]
patients_pd=pd.DataFrame(patients_rows,columns=["patient_id","patient_name","conditions"])
patients_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
patients_spark=spark.createDataFrame(patients_rows,"patient_id int, patient_name string, conditions string")
patients_spark.createOrReplaceTempView("Patients")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
SELECT patient_id, patient_name, conditions
FROM Patients
WHERE conditions LIKE 'DIAB1%'
   OR conditions LIKE '% DIAB1%'
ORDER BY patient_id
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
mask=patients_pd["conditions"].fillna("").str.contains(r"(?:^| )DIAB1",regex=True)
result_pd=patients_pd.loc[mask,["patient_id","patient_name","conditions"]].sort_values("patient_id").reset_index(drop=True)
result_pd

# 4. PySpark Solution

In [ ]:
result_spark=(patients_spark.filter(F.coalesce(F.col("conditions"),F.lit("")).rlike(r"(^| )DIAB1")).select("patient_id","patient_name","conditions").orderBy("patient_id"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| token-prefix match | two `LIKE` patterns | `.str.contains(regex)` | `.rlike(regex)` |
| null-safe string | SQL predicate ignores NULL | `.fillna("")` | `F.coalesce(..., "")` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Patients

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: patients_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: patients_spark